# Session 2 — Putting a p-value on a Network Property

**Goal of this session:** turn last session's one-off comparison into a function you can point at any metric, run it on several metrics at once, and handle what that does to your false-positive rate honestly.

*Network Neuroscience in Python, session 2 of 10.*

## Why this matters

Session 1 built one null distribution, for one metric, by hand. That doesn't scale — a real analysis usually reports several graph metrics side by side (clustering, path length, efficiency, modularity...), and copy-pasting the null-model loop for each one is exactly the kind of repetition that introduces bugs. We wrap it once, reuse it everywhere, and then confront the statistical cost of testing more than one thing.

## The toy network, again

Same function as session 1, pasted fresh so this notebook runs on its own.

In [ ]:
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt


def generate_toy_network(n_per_module=5, n_modules=2, p_within=0.7, p_between=0.05,
                          add_connector=True, connector_frac=0.6, seed=0):
    """A small synthetic multi-region network with known modular structure.
    See session 1 for the full explanation. Not a brain — a graph with a
    ground truth we invented ourselves.
    """
    rng = np.random.default_rng(seed)
    G = nx.Graph()
    node_id = 0
    modules = []
    for m in range(n_modules):
        members = []
        for _ in range(n_per_module):
            G.add_node(node_id, module=m)
            members.append(node_id)
            node_id += 1
        modules.append(members)
    for m in range(n_modules):
        members = modules[m]
        for i in range(len(members)):
            for j in range(i + 1, len(members)):
                if rng.random() < p_within:
                    G.add_edge(members[i], members[j])
    for m1 in range(n_modules):
        for m2 in range(m1 + 1, n_modules):
            for u in modules[m1]:
                for v in modules[m2]:
                    if rng.random() < p_between:
                        G.add_edge(u, v)
    if add_connector:
        connector = node_id
        G.add_node(connector, module="connector")
        n_link = max(1, round(connector_frac * n_per_module))
        for members in modules:
            chosen = rng.choice(members, size=min(n_link, len(members)), replace=False)
            for other in chosen:
                G.add_edge(connector, int(other))
    return G


G = generate_toy_network(seed=0)
print(G.number_of_nodes(), "nodes,", G.number_of_edges(), "edges")

## One function, any metric

`test_against_null` takes a graph and a metric function, builds an ensemble of degree-preserving randomisations (session 1), and returns everything you'd want to report: the observed value, the null mean and standard deviation, a z-score, and a two-sided permutation p-value.

We use a **two-sided** p-value here (checking whether the observed value is unusually high *or* unusually low), because unlike session 1 we're not committing in advance to a direction for every metric we test.

In [ ]:
def degree_preserving_null(G, n_swaps_per_edge=10, seed=0):
    rng = np.random.default_rng(seed)
    G_null = G.copy()
    n_swaps = max(10, n_swaps_per_edge * G.number_of_edges())
    nx.double_edge_swap(G_null, nswap=n_swaps, max_tries=n_swaps * 20,
                         seed=int(rng.integers(1_000_000_000)))
    return G_null


def test_against_null(G, metric_fn, n_random=500, n_swaps_per_edge=10, seed=0):
    """Compare metric_fn(G) against its degree-preserving null distribution.

    Returns a dict with the observed value, the null distribution itself,
    its mean and standard deviation, a z-score, and a two-sided
    permutation p-value.
    """
    rng = np.random.default_rng(seed)
    observed = metric_fn(G)
    null_vals = np.empty(n_random)
    for i in range(n_random):
        G_null = degree_preserving_null(G, n_swaps_per_edge=n_swaps_per_edge,
                                         seed=int(rng.integers(1_000_000_000)))
        null_vals[i] = metric_fn(G_null)

    null_mean, null_std = null_vals.mean(), null_vals.std()
    z = (observed - null_mean) / null_std if null_std > 0 else np.nan
    n_extreme = np.sum(np.abs(null_vals - null_mean) >= np.abs(observed - null_mean))
    p_perm = (n_extreme + 1) / (n_random + 1)

    return {
        "observed": observed,
        "null_values": null_vals,
        "null_mean": null_mean,
        "null_std": null_std,
        "z": z,
        "p": p_perm,
    }

## Three metrics at once

- **Clustering coefficient**, from session 1: local friend-of-a-friend density.
- **Characteristic path length** (`nx.average_shortest_path_length`): the average number of hops needed to get from one node to any other. Shorter means information can travel across the network more directly.
- **Global efficiency** (`nx.global_efficiency`): closely related to path length, but defined so that disconnected node pairs (infinite hops) contribute zero instead of breaking the average. It's often preferred for exactly that robustness.

All three get compared to their own degree-preserving null.

In [ ]:
metrics = {
    "clustering coefficient": nx.average_clustering,
    "characteristic path length": nx.average_shortest_path_length,
    "global efficiency": nx.global_efficiency,
}

results = {}
for name, fn in metrics.items():
    results[name] = test_against_null(G, fn, n_random=500, seed=hash(name) % 1000)
    r = results[name]
    print(f"{name:28s} observed={r['observed']:.3f}  z={r['z']:+.2f}  p={r['p']:.4f}")

## One null distribution per metric

Three small panels instead of one, each with its own null cloud and its own observed marker, so you can see at a glance which metrics look unusual and which don't.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, (name, r) in zip(axes, results.items()):
    ax.hist(r["null_values"], bins=25, color="#a0aec0", edgecolor="white")
    ax.axvline(r["observed"], color="#c53030", linewidth=3)
    ax.set_title(f"{name}\nz={r['z']:+.2f}, p={r['p']:.3f}", fontsize=11)
    ax.set_xlabel("null value", fontsize=10)
axes[0].set_ylabel("count", fontsize=10)
plt.tight_layout()
plt.show()

## The multiple-comparisons problem

Recall from session 9 of the first course: deciding what you're testing before you look matters, because running many tests inflates your chance of a false positive somewhere in the batch.

Here's the arithmetic. If each of these three tests uses the usual α = 0.05 cutoff, and — worst case — none of the underlying effects are real, the chance that *at least one* comes out "significant" by chance alone is not 5%. Assuming independence:

$$P(\text{at least one false positive}) = 1 - (1 - \alpha)^m$$

for `m` tests. That's the **family-wise error rate**, and it grows fast.

In [ ]:
alpha = 0.05
for m in [1, 3, 5, 10, 20]:
    fwer = 1 - (1 - alpha) ** m
    print(f"m={m:2d} tests  ->  chance of >=1 false positive = {fwer:.3f}")

## Two honest ways to handle it

**Bonferroni correction**: divide α by the number of tests, and only call something significant if it clears the stricter bar. Simple, conservative, easy to justify. For our three metrics, the corrected threshold is α / 3 ≈ 0.0167.

**Pre-registration / one hypothesis**: decide beforehand which single metric actually answers your question, test only that one, and report the others as description rather than inference. This is often the better choice in network neuroscience, where a graph produces dozens of correlated metrics and Bonferroni across all of them is needlessly punishing (the metrics aren't independent, so Bonferroni is actually *too* conservative in that case — a caveat worth knowing, even though we won't build the correction for correlated tests here).

We'll apply plain Bonferroni to our three metrics, because it's the safest default when you haven't checked how correlated your tests are.

In [ ]:
m = len(metrics)
alpha_corrected = alpha / m
print(f"uncorrected alpha: {alpha}")
print(f"Bonferroni-corrected alpha (m={m}): {alpha_corrected:.4f}\n")

for name, r in results.items():
    survives = r["p"] < alpha_corrected
    print(f"{name:28s} p={r['p']:.4f}   significant after correction: {survives}")

## Recap

You now have a single function, `test_against_null`, that turns "here's a graph metric" into "here's whether that metric is more extreme than a degree-matched null, with a z-score and a p-value" — and you know not to trust that p-value blindly when you've run it more than once.

**Next session:** instead of measuring properties of a network you already understand, we let an algorithm find structure in it that we didn't hand-specify — community detection.